In [20]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, f1_score, recall_score
import numpy as np
import random
from tqdm import tqdm

In [21]:
#фиксируем сиды
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [24]:
device = torch.device("cpu")

In [25]:
#базовые трансформации
baseline_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3), # Делаем 3 канала из ЧБ
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [26]:
#загрузка базового датасета
train_baseline_ds = datasets.ImageFolder('data/casting_data/casting_data/train', transform=baseline_transform)
test_baseline_ds = datasets.ImageFolder('data/casting_data/casting_data/test', transform=baseline_transform)

train_baseline_loader = DataLoader(train_baseline_ds, batch_size=16, shuffle=True)
test_baseline_loader = DataLoader(test_baseline_ds, batch_size=16, shuffle=False)


In [40]:
#фун-ия обучения и оценки
def train_and_evaluate(model, train_loader, test_loader, optimizer, criterion, epochs=5, scheduler=None):
    model.to(device)
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels in tqdm(train_loader, desc=f"epoch {epoch+1}/{epochs} [train]"):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            
        if scheduler:
            scheduler.step()
            
    #оценка
    model.eval()
    all_preds, all_labels = [],[]
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="[evaluate]"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    #класс 0-def_front(дефект),класс 1-ok_front
    f1 = f1_score(all_labels, all_preds, pos_label=0) # Важен класс дефекта
    recall = recall_score(all_labels, all_preds, pos_label=0)
    print(f"результаты: f1-score (defect) = {f1:.4f}, recall (defect) = {recall:.4f}\n")
    return f1, recall

In [28]:
print("\n---[ПУНКТ 2A, 2B] ОБУЧЕНИЕ БЕЙЗЛАЙНА (RESNET18) ---")
model_resnet_base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model_resnet_base.fc = nn.Linear(model_resnet_base.fc.in_features, 2)
optimizer_base = optim.Adam(model_resnet_base.parameters(), lr=1e-4)
train_and_evaluate(model_resnet_base, train_baseline_loader, test_baseline_loader, optimizer_base, nn.CrossEntropyLoss(), epochs=3)


---[ПУНКТ 2A, 2B] ОБУЧЕНИЕ БЕЙЗЛАЙНА (RESNET18) ---


[evaluate]: 100%|██████████| 45/45 [18:18<00:00, 24.40s/it]   


результаты: f1-score (defect) = 0.9978, recall (defect) = 0.9956



(0.997787610619469, 0.9955849889624724)

In [30]:
print("\n---[ПУНКТ 2A, 2B] ОБУЧЕНИЕ БЕЙЗЛАЙНА (ViT) ---")
model_vit_base = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
model_vit_base.heads.head = nn.Linear(model_vit_base.heads.head.in_features, 2)
optimizer_vit_base = optim.Adam(model_vit_base.parameters(), lr=1e-4)
train_and_evaluate(model_vit_base, train_baseline_loader, test_baseline_loader, optimizer_vit_base, nn.CrossEntropyLoss(), epochs=3)


---[ПУНКТ 2A, 2B] ОБУЧЕНИЕ БЕЙЗЛАЙНА (ViT) ---


[evaluate]: 100%|██████████| 45/45 [00:58<00:00,  1.29s/it]

результаты: f1-score (defect) = 0.9923, recall (defect) = 0.9912



(0.9922651933701657, 0.9911699779249448)

In [35]:
#3) улучшение бэйзлайна(гипотезы)

#гипотеза 1:используем чистые данные 512x512 и свои аугментации
improved_transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

#применяем трансформации к сырому датасету (разбиваем на train/test)
raw_dataset = datasets.ImageFolder('data/casting_512x512/casting_512x512', transform=improved_transform_train)
train_size = int(0.8 * len(raw_dataset))
test_size = len(raw_dataset) - train_size
train_improved_ds, test_improved_ds = torch.utils.data.random_split(raw_dataset, [train_size, test_size])

train_improved_loader = DataLoader(train_improved_ds, batch_size=16, shuffle=True)
test_improved_loader = DataLoader(test_improved_ds, batch_size=16, shuffle=False)

In [36]:
print("\n--- [ПУНКТ 3D, 3E] УЛУЧШЕННЫЙ БЕЙЗЛАЙН ---")
model_resnet_imp = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model_resnet_imp.fc = nn.Linear(model_resnet_imp.fc.in_features, 2)
#гипотеза 2: AdamW + Scheduler
optimizer_imp = optim.AdamW(model_resnet_imp.parameters(), lr=3e-4, weight_decay=0.01)
scheduler_imp = optim.lr_scheduler.CosineAnnealingLR(optimizer_imp, T_max=5)
train_and_evaluate(model_resnet_imp, train_improved_loader, test_improved_loader, optimizer_imp, nn.CrossEntropyLoss(), epochs=5, scheduler=scheduler_imp)


--- [ПУНКТ 3D, 3E] УЛУЧШЕННЫЙ БЕЙЗЛАЙН ---


[evaluate]: 100%|██████████| 17/17 [00:10<00:00,  1.55it/s]

результаты: f1-score (defect) = 0.9937, recall (defect) = 0.9874



(0.9936708860759493, 0.9874213836477987)

In [37]:
#4)самописная модель

class CustomCNN(nn.Module):
    def __init__(self):
        super(CustomCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), #112x112
            
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), #56x56
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  #28x28
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 28 * 28, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


In [38]:
print("\n--- [ПУНКТ 4B, 4C] САМОПИСНАЯ МОДЕЛЬ (БАЗОВЫЕ ДАННЫЕ) ---")
custom_model_base = CustomCNN()
opt_custom_base = optim.Adam(custom_model_base.parameters(), lr=1e-3)
train_and_evaluate(custom_model_base, train_baseline_loader, test_baseline_loader, opt_custom_base, nn.CrossEntropyLoss(), epochs=5)


--- [ПУНКТ 4B, 4C] САМОПИСНАЯ МОДЕЛЬ (БАЗОВЫЕ ДАННЫЕ) ---


[evaluate]: 100%|██████████| 45/45 [00:09<00:00,  4.80it/s]

результаты: f1-score (defect) = 0.9787, recall (defect) = 0.9647



(0.9787234042553192, 0.9646799116997793)

In [39]:
print("\n--- [ПУНКТ 4G, 4H] САМОПИСНАЯ МОДЕЛЬ ---")
custom_model_imp = CustomCNN()
opt_custom_imp = optim.AdamW(custom_model_imp.parameters(), lr=1e-3, weight_decay=0.01)
sched_custom_imp = optim.lr_scheduler.CosineAnnealingLR(opt_custom_imp, T_max=5)
train_and_evaluate(custom_model_imp, train_improved_loader, test_improved_loader, opt_custom_imp, nn.CrossEntropyLoss(), epochs=5, scheduler=sched_custom_imp)


--- [ПУНКТ 4G, 4H] САМОПИСНАЯ МОДЕЛЬ ---


[evaluate]: 100%|██████████| 17/17 [00:03<00:00,  5.40it/s]

результаты: f1-score (defect) = 0.8281, recall (defect) = 0.7421



(0.8280701754385964, 0.7421383647798742)

In [46]:
print("ИТОГИ")
print("\n" + "="*50)
print("отчет по лабораторной работе")
print("="*50)

print("\n1. выбор условий")
print("взял датасет casting defect image data. это реальная задача для заводов — искать брак в деталях.")
print("метрики выбрал f1-score и recall. recall тут важнее всего, чтобы не пропустить дефектную деталь.")

print("\n2. результаты бейзлайна")
print("обучил resnet18 и vit_b_16 на базовых данных.")
print("- resnet18: f1 = 0.9978, recall = 0.9956")
print("- vit: f1 = 0.9923, recall = 0.9912")
print("вывод: реснет сработал чуть лучше, свертки хорошо цепляют мелкие трещины на металле.")

print("\n3. улучшение бейзлайна")
print("гипотеза: свои аугментации на чистых данных 512x512 сделают модель надежнее. еще сменил адам на адамв.")
print("результат: f1 = 0.9937, recall = 0.9874")
print("вывод: цифры стали чуть ниже, зато модель теперь честная и не зазубривает шумы из папки трейна.")

print("\n4. самописная модель")
print("собрал cnn из 3 блоков (conv + batchnorm + relu) и классификатора.")
print("- на базовых данных: f1 = 0.9787 (хуже реснета, так как нет предобучения)")
print("- на чистых данных с техниками: f1 = 0.8281")
print("вывод: падение метрик на чистых данных связано с тем, что их физически меньше (~1300 штук),")
print("для обучения с нуля такой сложной текстуры этого маловато.")

print("\nитоговый вывод:")
print("для реального производства лучше всего подходит resnet18. она стабильная и точная.")
print("свою модель можно использовать только на очень слабом железе для скорости.")
print("="*50)

ИТОГИ

отчет по лабораторной работе

1. выбор условий
взял датасет casting defect image data. это реальная задача для заводов — искать брак в деталях.
метрики выбрал f1-score и recall. recall тут важнее всего, чтобы не пропустить дефектную деталь.

2. результаты бейзлайна
обучил resnet18 и vit_b_16 на базовых данных.
- resnet18: f1 = 0.9978, recall = 0.9956
- vit: f1 = 0.9923, recall = 0.9912
вывод: реснет сработал чуть лучше, свертки хорошо цепляют мелкие трещины на металле.

3. улучшение бейзлайна
гипотеза: свои аугментации на чистых данных 512x512 сделают модель надежнее. еще сменил адам на адамв.
результат: f1 = 0.9937, recall = 0.9874
вывод: цифры стали чуть ниже, зато модель теперь честная и не зазубривает шумы из папки трейна.

4. самописная модель
собрал cnn из 3 блоков (conv + batchnorm + relu) и классификатора.
- на базовых данных: f1 = 0.9787 (хуже реснета, так как нет предобучения)
- на чистых данных с техниками: f1 = 0.8281
вывод: падение метрик на чистых данных связано с 